### Task 1: Build a Complete ETL Pipeline
Build an ETL pipeline that processes raw e-commerce order data into a clean analytical dataset.

Step 1 — Generate raw data: Create a list of 200 raw order records as dictionaries with these fields: order_id, customer_id, product_name, quantity, unit_price, order_date, shipping_country. Intentionally include at least 15 problematic records across these categories:

3 records with missing product_name (None or empty string)

3 records with negative quantity or unit_price

3 records with malformed order_date (e.g., "not-a-date", "2025-13-45")

3 records with duplicate order_id values

3 records where quantity or unit_price is a string instead of a number

In [1]:
#Step 1
import random
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

products = ["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones"]
countries = ["Azerbaijan", "Germany", "France", "Canada", "Japan"]

raw_records = []

for i in range(200):
    
    record = {
        "order_id": i,
        "customer_id": random.randint(1,20),
        "product_name": random.choice(products),
        "quantity": random.randint(1,5),
        "unit_price": round(random.uniform(5,200),2),
        "order_date": str(datetime(2025,1,1) + timedelta(days=random.randint(0,90))),
        "shipping_country": random.choice(countries)
    }
    
    raw_records.append(record)

In this step, we simulate a raw dataset that represents e-commerce order records.  
The goal is to create synthetic data that will later be used in the ETL pipeline.

Product and Country Lists
First, we define two lists:

- **products** → a list of available product names
- **countries** → a list of possible shipping destinations

These lists will be used to randomly assign values to each generated order.

```python
products = ["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones"]
countries = ["usa", "germany", "france", "canada", "japan"]

In [2]:
raw_records[0]["product_name"] = ""
raw_records[1]["product_name"] = None
raw_records[2]["product_name"] = None

raw_records[3]["quantity"] = -3
raw_records[4]["unit_price"] = -10
raw_records[5]["quantity"] = -2

raw_records[6]["order_date"] = "2025-13-43"
raw_records[7]["order_date"] = "2025-13-45"
raw_records[8]["order_date"] = "not-a-date"

raw_records[9]["order_id"] = raw_records[19]["order_id"]
raw_records[10]["order_id"] = raw_records[18]["order_id"]
raw_records[11]["order_id"] = raw_records[23]["order_id"]

raw_records[12]["quantity"] = "nine"
raw_records[13]["unit_price"] = "twenty one"
raw_records[14]["quantity"] = "one"



For testing the extract function, we are deliberately modifying some records to be invalid.  


In [3]:
from collections import Counter
def extract(raw_records):

    valid_records = []
    rejected_records = []
    order_ids = set()

    for record in raw_records:

        try:

            required_fields = [
                "order_id","customer_id","product_name",
                "quantity","unit_price","order_date","shipping_country"
            ]

            if not all(field in record for field in required_fields):
                rejected_records.append({"record": record, "reason": "missing field"})
                continue

            if record["product_name"] is None or record["product_name"] == "":
                rejected_records.append({"record": record, "reason": "missing product_name"})
                continue

            quantity = float(record["quantity"])
            price = float(record["unit_price"])

            if quantity <= 0 or price <= 0:
                rejected_records.append({"record": record, "reason": "negative quantity or price"})
                continue

            try:
                pd.to_datetime(record["order_date"])
            
            except:
                rejected_records.append({"record": record, "reason": "invalid order_date"})
                continue

            if record["order_id"] in order_ids:
                rejected_records.append({"record": record, "reason": "duplicate order_id"})
                continue

            order_ids.add(record["order_id"])
            valid_records.append(record)

        except:
            rejected_records.append({"record": record, "reason": "invalid data type"})

    reason_counts = Counter(r["reason"] for r in rejected_records)

    print("Valid records:", len(valid_records))
    print("Rejected records:", len(rejected_records))

    for reason, count in reason_counts.items():
        print(reason, ":", count)

    return valid_records, rejected_records
extract(raw_records)

Valid records: 185
Rejected records: 15
missing product_name : 3
negative quantity or price : 3
invalid order_date : 3
invalid data type : 3
duplicate order_id : 3


([{'order_id': 19,
   'customer_id': 8,
   'product_name': 'Laptop',
   'quantity': 5,
   'unit_price': 188.63,
   'order_date': '2025-01-09 00:00:00',
   'shipping_country': 'Japan'},
  {'order_id': 18,
   'customer_id': 20,
   'product_name': 'Monitor',
   'quantity': 2,
   'unit_price': 117.72,
   'order_date': '2025-02-26 00:00:00',
   'shipping_country': 'Canada'},
  {'order_id': 23,
   'customer_id': 13,
   'product_name': 'Mouse',
   'quantity': 5,
   'unit_price': 49.09,
   'order_date': '2025-02-18 00:00:00',
   'shipping_country': 'Canada'},
  {'order_id': 15,
   'customer_id': 2,
   'product_name': 'Keyboard',
   'quantity': 3,
   'unit_price': 199.72,
   'order_date': '2025-03-07 00:00:00',
   'shipping_country': 'Japan'},
  {'order_id': 16,
   'customer_id': 1,
   'product_name': 'Keyboard',
   'quantity': 4,
   'unit_price': 192.55,
   'order_date': '2025-01-05 00:00:00',
   'shipping_country': 'France'},
  {'order_id': 17,
   'customer_id': 19,
   'product_name': 'Laptop


In this step, we will check the `raw_records` list and validate each record:

- Ensure all required fields exist
- Make sure `quantity` and `unit_price` are positive numbers
- Check that `order_date` is a parseable date

Each rejected record will include the reason for rejection, and at the end, a summary will be printed.

---

In [4]:
#Step 3  — Transform: Write a transform(valid_records) function that:
import pandas as pd

def transform(valid_records):

    df = pd.DataFrame(valid_records)

    df["order_date"] = pd.to_datetime(df["order_date"])

    df["quantity"] = df["quantity"].astype(float)
    df["unit_price"] = df["unit_price"].astype(float)

    df["total_amount"] = df["quantity"] * df["unit_price"]

    df["order_month"] = df["order_date"].dt.to_period("M")

    df["order_day_of_week"] = df["order_date"].dt.day_name()

    df["shipping_country"] = df["shipping_country"].str.title()

    df = df.drop_duplicates(subset="order_id", keep="first")

    def categorize(amount):
        if amount < 25:
            return "small"
        elif amount <= 100:
            return "medium"
        else:
            return "large"

    df["amount_category"] = df["total_amount"].apply(categorize)

    return df

- In this step, we convert the `valid_records` into a clean analytical dataset.  
- We first convert `order_date` to datetime and ensure `quantity` and `unit_price` are floats.  
- Next, we calculate `total_amount` and extract the order month and day of the week.  
- Shipping country names are standardized, and duplicate orders are removed based on `order_id`.  
- Finally, each order is categorized as "small", "medium", or "large" based on its total amount.  
- The resulting dataset is ready for analysis and visualization.
---

In [5]:
#Step 4 — Load: Write a load(df, path) function that saves
#the DataFrame to a Parquet file. Then read it back and verify the row count and dtypes match.

def load(df, path):
    df.to_parquet(path)
    loaded_df = pd.read_parquet(path)
    print(len(df))
    print(len(loaded_df))
    print(df.dtypes)
    print(loaded_df.dtypes)
    return loaded_df

valid_records, rejected_records = extract(raw_records)
df = transform(valid_records)
loaded_df = load(df, "orders.parquet")

print("Total raw records:", len(raw_records))
print("Valid records after extraction:", len(valid_records))
print("Cleaned DataFrame after transform:", len(df))
print("Loaded DataFrame from Parquet:", len(loaded_df))
print("Rejected records during extraction:", len(rejected_records))

Valid records: 185
Rejected records: 15
missing product_name : 3
negative quantity or price : 3
invalid order_date : 3
invalid data type : 3
duplicate order_id : 3
185
185
order_id                      int64
customer_id                   int64
product_name                 object
quantity                    float64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month               period[M]
order_day_of_week            object
amount_category              object
dtype: object
order_id                      int64
customer_id                   int64
product_name                 object
quantity                    float64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month               period[M]
order_day_of_week            object
amount_category              object
dtype: object
Total ra


In this step, we save the transformed DataFrame to a Parquet file and verify it was saved correctly.  

The `load(df, path)` function writes the DataFrame to the given path, reads it back, and prints the row counts and data types to ensure they match.  

After loading, we also print a summary including:  
- Total raw records  
- Valid records after extraction  
- Cleaned DataFrame rows after transformation  
- Loaded DataFrame rows from Parquet  
- Rejected records during extraction  

This ensures the ETL pipeline preserves data integrity and can be used for further analysis.

---

In [6]:
#Step 5 Run the full pipeline: 
valid_records, rejected_records = extract(raw_records)

df_clean = transform(valid_records)
print("-"*50)
df_loaded = load(df_clean,"orders_clean.parquet")
print("-"*50)
print("Raw records:", len(raw_records))
print("Valid records:", len(valid_records))
print("Transformed rows:", len(df_clean))
print("Loaded rows:", len(df_loaded))

Valid records: 185
Rejected records: 15
missing product_name : 3
negative quantity or price : 3
invalid order_date : 3
invalid data type : 3
duplicate order_id : 3
--------------------------------------------------
185
185
order_id                      int64
customer_id                   int64
product_name                 object
quantity                    float64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month               period[M]
order_day_of_week            object
amount_category              object
dtype: object
order_id                      int64
customer_id                   int64
product_name                 object
quantity                    float64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month               period[M]
order_day_of_week            object
amount_

In this step, we run the entire ETL process from start to finish:

1. **Extract:** We separate `raw_records` into `valid_records` and `rejected_records`.  
2. **Transform:** We clean and enhance the valid records into a structured DataFrame (`df_clean`) with additional columns such as `total_amount`, `order_month`, `order_day_of_week`, and `amount_category`.  
3. **Load:** We save the cleaned DataFrame to a Parquet file and read it back to verify that the row count and data types match.  
4. **Summary:** Finally, we print a summary showing the number of records at each stage: raw, valid, transformed, and loaded.

---

### Task 2: ETL vs ELT Comparison
Step 1: Implement an ELT variant of the pipeline from Task 1:

Extract: do minimal validation (only reject unparseable JSON/records)

Load: save the raw (mostly unvalidated) data to a Parquet "data lake" file

Transform: read from the data lake file, apply the same validation and transformation rules as Task 1

Step 2: Compare ETL and ELT approaches. Write a markdown cell addressing:

How many records made it to the destination in each approach?

At what stage were problems caught in each?

What are the advantages of each approach?

When would you choose one over the other?

In an **ELT (Extract, Load, Transform)** pipeline, the first step is to **extract the raw data from the source system** without applying any transformations or validations.

Unlike ETL pipelines, ELT pipelines typically **load raw data directly into storage (such as a data lake or warehouse)** and perform transformations later.


In [7]:
#Step 1
import pandas as pd
from collections import Counter

raw_records_elt = []
rejected_first = []

for record in raw_records:
    try:
        raw_records_elt.append(record)
    except:
        rejected_first.append(record)

In this step, we perform a *minimal extract* of the raw records for the ELT pipeline.  

- We read each record from the `raw_records` list.  
- Since this is **minimal validation**, we only reject records that cannot be read or parsed at all.  
- All readable records are appended to `raw_records_elt` for loading into the data lake.  
- Any completely unparseable records are added to `rejected_first`.
---

In [8]:
df_raw = pd.DataFrame(raw_records_elt)
df_raw = df_raw.astype(str)  
df_raw.to_parquet("data_lake_orders_raw.parquet", engine="pyarrow")

df_parquet = pd.read_parquet("data_lake_orders_raw.parquet")

In this step, we save the minimally validated raw records to a Parquet file in the data lake.  
- We first convert `raw_records_elt` into a DataFrame and cast all columns to strings for consistent storage.  
- Then, we write the DataFrame to `"data_lake_orders_raw.parquet"` using the `pyarrow` engine.  
- After saving, we read the Parquet file back to verify that all records were stored correctly.  
- This ensures the raw data is preserved for later transformations while keeping the pipeline simple and reproducible.
---

In [9]:
valid_records = []
rejected_records = []
order_ids = set()

for record in df_parquet.to_dict(orient="records"):
    try:
        required_fields = [
            "order_id","customer_id","product_name",
            "quantity","unit_price","order_date","shipping_country"]

        if not all(field in record for field in required_fields):
            rejected_records.append({"record": record, "reason": "missing field"})
            continue

        
        if record["product_name"] in [None,"","None"]:
            rejected_records.append({"record": record, "reason": "missing product_name"})
            continue

        
        quantity = float(record["quantity"])
        price = float(record["unit_price"])
        if quantity <= 0 or price <= 0:
            rejected_records.append({"record": record, "reason": "negative quantity or price"})
            continue

        
        try:
            date = pd.to_datetime(record["order_date"], errors="coerce")
            if pd.isna(date):
                rejected_records.append({"record": record, "reason": "invalid order_date"})
                continue
        except:
            rejected_records.append({"record": record, "reason": "invalid order_date"})
            continue

        
        if record["order_id"] in order_ids:
            rejected_records.append({"record": record, "reason": "duplicate order_id"})
            continue

        order_ids.add(record["order_id"])

        
        record["quantity"] = quantity
        record["unit_price"] = price
        record["order_date"] = date

        valid_records.append(record)

    except:
        rejected_records.append({"record": record, "reason": "invalid data type"})



reason_counts = Counter(r["reason"] for r in rejected_records)
print("Valid records:", len(valid_records))
print("Rejected records:", len(rejected_records))
for reason, count in reason_counts.items():
    print(reason, ":", count)


df_clean = pd.DataFrame(valid_records)

df_clean["total_amount"] = df_clean["quantity"] * df_clean["unit_price"]
df_clean["order_month"] = df_clean["order_date"].dt.to_period("M")
df_clean["order_day_of_week"] = df_clean["order_date"].dt.day_name()
df_clean["shipping_country"] = df_clean["shipping_country"].str.title()

df_clean = df_clean.drop_duplicates(subset="order_id", keep="first")

df_clean["amount_category"] = pd.cut(
    df_clean["total_amount"],
    bins=[0,25,100,float("inf")],
    labels=["small","medium","large"]
)

print("Transformed rows:", len(df_clean))

df_clean.to_parquet("data_lake_orders_clean.parquet", engine="pyarrow")

df_loaded = pd.read_parquet("data_lake_orders_clean.parquet")
print("Loaded rows:", len(df_loaded))
print(df_loaded.dtypes)

Valid records: 185
Rejected records: 15
missing product_name : 3
negative quantity or price : 3
invalid order_date : 3
invalid data type : 3
duplicate order_id : 3
Transformed rows: 185
Loaded rows: 185
order_id                     object
customer_id                  object
product_name                 object
quantity                    float64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month               period[M]
order_day_of_week            object
amount_category            category
dtype: object



In this step, we read the raw data from the Parquet data lake and validate each record, checking for missing fields, invalid product names, negative quantities or prices, unparseable dates, and duplicate order IDs.  
Valid records are cleaned and stored in `valid_records`, while invalid ones are stored in `rejected_records` with a reason for rejection.  
We then convert the valid records into a DataFrame, calculate `total_amount`, extract `order_month` and `order_day_of_week`, and standardize `shipping_country` to title case.  
Duplicates are removed based on `order_id`, and orders are categorized into "small", "medium", or "large" based on `total_amount`.  
Finally, the cleaned DataFrame is saved to a new Parquet file, read back to verify, and the row counts and data types are printed to ensure data integrity.  
This completes the transformation and loading of the cleaned dataset from the raw data lake.

---

In [10]:
#Step 2

#### How many records made it to the destination in each approach?


In both approaches, 15 records were lost. The summary is shown below:

| Approach | Records Extracted | Records Lost / Problematic | Records Loaded to Destination | Problem Stage |
|----------|-----------------|---------------------------|-----------------------------|---------------|
| ETL      | 200             | 15                        | 185                         | Transformation |
| ELT      | 200             | 15                        | 185                         | After Load / Transformation |

**Explanation:**

- **ETL:** The problematic records were caught during the **transformation** stage, so they were not loaded into the destination.  
- **ELT:** The problematic records were identified during or after loading, in the **transformation stage at the destination**, resulting in the same number of lost records.  

This comparison highlights how ETL and ELT differ in **when and where data quality issues are detected**.

#### At what stage were problems caught in each?

- **ETL:** Errors are detected during the **transformation** stage, meaning problematic data is **never loaded** into the destination.
- **ELT:** Errors are detected in the **destination** during or after the transformation stage, meaning problematic data is **loaded first**.


#### What are the advantages of each approach?


**ETL Advantages:**
- Only clean and correct data is loaded → less storage required and lower risk.
- Complex transformations can be performed outside the destination server.

**ELT Advantages:**
- Large volumes of data can be quickly loaded into cloud storage.
- Transformations can be executed in parallel and faster using SQL or analytics engines.

####  When would you choose one over the other?


**ETL:**  
ETL is preferred when data quality is critical. It ensures that only clean and validated data reaches the destination. Transformations are handled before loading, which reduces the risk of errors in the data warehouse. This approach works well when resources for transformation can be managed during the extract stage. ETL is also suitable for on-premise systems or smaller datasets where performance is manageable.

**ELT:**  
ELT is preferred in cloud environments and large data warehouses. It allows fast loading of raw data into the destination. Transformations are performed inside the destination, leveraging the power of modern analytics engines. This approach provides flexibility and scalability for handling large datasets. ELT is ideal when you need parallel processing and dynamic transformation capabilities.



### Task 3: Simulate Modes of Dataflow
Model a simplified analytics system with three components: an order ingestion service, a feature computation service, and a prediction service.


In [11]:
#Step 1 — Data Passing Through a Database
import pandas as pd
from datetime import datetime, timedelta
import random

database = {"orders": [], "features": {}}

def order_service_write(order):
    database["orders"].append(order)

def feature_service_compute():
    if not database["orders"]:
        return
    df = pd.DataFrame(database["orders"])
    features = df.groupby("customer_id").agg(
        total_orders=("order_id", "count"),
        avg_amount=("total_amount", "mean"),
        last_order=("order_date", "max")
    )
    database["features"] = features.to_dict("index")

def prediction_service_read(customer_id):
    return database["features"].get(customer_id, None)

In [12]:
# Task 3 Step 2 — Data passing through services using Task 1 ETL output
import pandas as pd

class OrderService:
    def __init__(self, orders_df):
        self.orders_df = orders_df

    def get_orders(self):
        return self.orders_df.to_dict(orient="records")


class FeatureService:
    def __init__(self, order_service):
        self.order_service = order_service
        self.features = {}

    def compute_features(self):
        orders = self.order_service.get_orders()  # "API call" to order service
        customer_data = {}
        for order in orders:
            cid = order["customer_id"]
            customer_data.setdefault(cid, {"total_orders": 0, "total_amount": 0.0, "last_order_date": None})
            customer_data[cid]["total_orders"] += 1
            customer_data[cid]["total_amount"] += order["quantity"] * order["unit_price"]
            order_date = pd.to_datetime(order["order_date"], errors="coerce")
            if order_date and (customer_data[cid]["last_order_date"] is None or order_date > customer_data[cid]["last_order_date"]):
                customer_data[cid]["last_order_date"] = order_date
        for cid, data in customer_data.items():
            data["avg_amount"] = data["total_amount"] / data["total_orders"]
        self.features = customer_data
        return self.features


class PredictionService:
    def __init__(self, feature_service):
        self.feature_service = feature_service

    def get_customer_features(self, customer_id):
        features = self.feature_service.compute_features()  # "API call" to feature service
        return features.get(customer_id, None)

order_service = OrderService(df_clean)
feature_service = FeatureService(order_service)
prediction_service = PredictionService(feature_service)

sample_customers = df_clean["customer_id"].drop_duplicates().sample(5, random_state=42).tolist()
for cid in sample_customers:
    customer_features = prediction_service.get_customer_features(cid)
    print(f"Customer {cid} features:\n{customer_features}\n")

Customer 8 features:
{'total_orders': 11, 'total_amount': 3840.31, 'last_order_date': Timestamp('2025-03-13 00:00:00'), 'avg_amount': 349.1190909090909}

Customer 9 features:
{'total_orders': 11, 'total_amount': 5098.580000000001, 'last_order_date': Timestamp('2025-03-09 00:00:00'), 'avg_amount': 463.5072727272728}

Customer 5 features:
{'total_orders': 10, 'total_amount': 3001.56, 'last_order_date': Timestamp('2025-03-27 00:00:00'), 'avg_amount': 300.156}

Customer 20 features:
{'total_orders': 12, 'total_amount': 4836.21, 'last_order_date': Timestamp('2025-03-29 00:00:00'), 'avg_amount': 403.0175}

Customer 3 features:
{'total_orders': 10, 'total_amount': 4278.92, 'last_order_date': Timestamp('2025-03-26 00:00:00'), 'avg_amount': 427.892}



The goal of this step is to simulate data flow between services without using a shared database. Instead of writing and reading from a central storage, each service communicates directly with the next one through function calls, mimicking real-world API requests. 

- The Order Service holds all cleaned orders from the ETL pipeline and provides them on request.
- The Feature Service fetches orders from the Order Service and computes aggregated features for each customer, such as total orders, total amount spent, and the date of the last order.
- The Prediction Service requests customer features from the Feature Service, simulating how a downstream system would consume analytics or machine learning inputs.

---

In [22]:
# Step 3 — Data passing through a message broker
import pandas as pd
from collections import defaultdict

class MessageBroker:
    def __init__(self):
        self.topics = defaultdict(list)
        self.subscribers = defaultdict(list)

    def publish(self, topic, message):
        self.topics[topic].append(message)
        for callback in self.subscribers[topic]:
            callback(message)

    def subscribe(self, topic, callback):
        self.subscribers[topic].append(callback)

    def get_latest(self, topic):
        if self.topics[topic]:
            return self.topics[topic][-1]
        return None


class OrderService:
    def __init__(self, broker):
        self.broker = broker

    def add_order(self, order):
        self.broker.publish("orders", order)


class FeatureService:
    def __init__(self, broker):
        self.broker = broker
        self.features = {}
        self.broker.subscribe("orders", self.process_order)

    def process_order(self, order):
        cid = order["customer_id"]

        self.features.setdefault(cid, {
            "total_orders": 0,
            "total_amount": 0.0,
            "last_order_date": None
        })

        self.features[cid]["total_orders"] += 1
        self.features[cid]["total_amount"] += order["quantity"] * order["unit_price"]

        order_date = pd.to_datetime(order["order_date"], errors="coerce")

        if order_date and (
            self.features[cid]["last_order_date"] is None
            or order_date > self.features[cid]["last_order_date"]
        ):
            self.features[cid]["last_order_date"] = order_date

        self.broker.publish("features", {cid: self.features[cid]})


class PredictionService:
    def __init__(self, broker):
        self.broker = broker
        self.customer_features = {}
        self.broker.subscribe("features", self.update_features)

    def update_features(self, features_update):
        self.customer_features.update(features_update)

    def get_customer_features(self, customer_id):
        return self.customer_features.get(customer_id, None)


broker = MessageBroker()
order_service = OrderService(broker)
feature_service = FeatureService(broker)
prediction_service = PredictionService(broker)

for i, order in enumerate(df_clean.sample(20, random_state=42).to_dict(orient="records"), 1):
    order_service.add_order(order)

sample_customers = df_clean["customer_id"].drop_duplicates().sample(5, random_state=42).tolist()

for cid in sample_customers:
    features = prediction_service.get_customer_features(cid)

    if features:
        avg_amount = features["total_amount"] / features["total_orders"]
        print(f"Customer {cid} features:")
        print(features)
        print(f"avg_amount: {avg_amount}\n")
    else:
        print(f"Customer {cid} features: None\n")

Customer 8 features:
{'total_orders': 1, 'total_amount': 80.21, 'last_order_date': Timestamp('2025-01-12 00:00:00')}
avg_amount: 80.21

Customer 9 features:
{'total_orders': 1, 'total_amount': 612.0, 'last_order_date': Timestamp('2025-01-09 00:00:00')}
avg_amount: 612.0

Customer 5 features:
{'total_orders': 2, 'total_amount': 617.54, 'last_order_date': Timestamp('2025-01-30 00:00:00')}
avg_amount: 308.77

Customer 20 features:
{'total_orders': 2, 'total_amount': 568.75, 'last_order_date': Timestamp('2025-03-26 00:00:00')}
avg_amount: 284.375

Customer 3 features:
{'total_orders': 1, 'total_amount': 536.64, 'last_order_date': Timestamp('2025-01-29 00:00:00')}
avg_amount: 536.64



The goal of this step is to simulate **event-driven data flow** using a **publish-subscribe (pub/sub) pattern**, which is common in modern analytics and microservice architectures. Instead of direct service-to-service calls or a shared database, services communicate by **publishing messages to topics** and **subscribing to topics of interest**.  

- The **Order Service** publishes new orders to the `"orders"` topic.  
- The **Feature Service** subscribes to `"orders"`, processes incoming orders, computes per-customer features, and publishes the results to the `"features"` topic.  
- The **Prediction Service** subscribes to `"features"` and keeps an up-to-date view of customer features.  

This setup demonstrates several key points:

1. **Loose coupling**: Services don’t need to know about each other’s internal implementation. They only interact via topics.  
2. **Real-time processing**: As soon as a new order arrives, features are updated and made available downstream immediately.  
3. **Resilience**: If one service goes down temporarily, it can catch up on messages once it comes back online (depending on the broker implementation).  
4. **Scalability**: Multiple instances of a service can subscribe to the same topic to handle high throughput.  

---


In [23]:
# Task 3 Step 4 — Run 20 new orders through all three dataflow modes
import pandas as pd
from collections import defaultdict

new_orders = df_clean.sample(20, random_state=42).to_dict(orient="records")
sample_customers = df_clean["customer_id"].drop_duplicates().sample(5, random_state=42).tolist()

database = {"orders": [], "features": {}}

def db_order_service_write(order):
    database["orders"].append(order)

def db_feature_service_compute():
    customer_data = {}
    for order in database["orders"]:
        cid = order["customer_id"]
        customer_data.setdefault(cid, {"total_orders": 0, "total_amount": 0.0, "last_order_date": None})
        
        customer_data[cid]["total_orders"] += 1
        customer_data[cid]["total_amount"] += order["quantity"] * order["unit_price"]
        
        order_date = pd.to_datetime(order["order_date"], errors="coerce")
        
        if order_date and (customer_data[cid]["last_order_date"] is None or order_date > customer_data[cid]["last_order_date"]):
            customer_data[cid]["last_order_date"] = order_date

    for cid in customer_data:
        total_orders = customer_data[cid]["total_orders"]
        total_amount = customer_data[cid]["total_amount"]
        customer_data[cid]["avg_amount"] = total_amount / total_orders if total_orders > 0 else 0

    database["features"] = customer_data


def db_prediction_service_read(customer_id):
    return database["features"].get(customer_id, None)


for order in new_orders:
    db_order_service_write(order)

db_feature_service_compute()

db_results = {cid: db_prediction_service_read(cid) for cid in sample_customers}



class OrderService:
    def __init__(self):
        self.orders = []

    def add_order(self, order):
        self.orders.append(order)

    def get_orders(self):
        return self.orders


class FeatureService:
    def __init__(self, order_service):
        self.order_service = order_service
        self.features = {}

    def compute_features(self):
        orders = self.order_service.get_orders()
        customer_data = {}

        for order in orders:
            cid = order["customer_id"]

            customer_data.setdefault(cid, {"total_orders": 0, "total_amount": 0.0, "last_order_date": None})

            customer_data[cid]["total_orders"] += 1
            customer_data[cid]["total_amount"] += order["quantity"] * order["unit_price"]

            order_date = pd.to_datetime(order["order_date"], errors="coerce")

            if order_date and (customer_data[cid]["last_order_date"] is None or order_date > customer_data[cid]["last_order_date"]):
                customer_data[cid]["last_order_date"] = order_date

        for cid in customer_data:
            total_orders = customer_data[cid]["total_orders"]
            total_amount = customer_data[cid]["total_amount"]
            customer_data[cid]["avg_amount"] = total_amount / total_orders if total_orders > 0 else 0

        self.features = customer_data
        return self.features


class PredictionService:
    def __init__(self, feature_service):
        self.feature_service = feature_service

    def get_customer_features(self, customer_id):
        features = self.feature_service.compute_features()
        return features.get(customer_id, None)


order_service = OrderService()
feature_service = FeatureService(order_service)
prediction_service = PredictionService(feature_service)

for order in new_orders:
    order_service.add_order(order)

ss_results = {cid: prediction_service.get_customer_features(cid) for cid in sample_customers}



class MessageBroker:
    def __init__(self):
        self.topics = defaultdict(list)
        self.subscribers = defaultdict(list)

    def publish(self, topic, message):
        self.topics[topic].append(message)

        for callback in self.subscribers[topic]:
            callback(message)

    def subscribe(self, topic, callback):
        self.subscribers[topic].append(callback)

    def get_latest(self, topic):
        if self.topics[topic]:
            return self.topics[topic][-1]
        return None


class OrderServiceBroker:
    def __init__(self, broker):
        self.broker = broker

    def add_order(self, order):
        self.broker.publish("orders", order)


class FeatureServiceBroker:
    def __init__(self, broker):
        self.broker = broker
        self.features = {}
        self.broker.subscribe("orders", self.process_order)

    def process_order(self, order):
        cid = order["customer_id"]

        self.features.setdefault(cid, {"total_orders": 0, "total_amount": 0.0, "last_order_date": None})

        self.features[cid]["total_orders"] += 1
        self.features[cid]["total_amount"] += order["quantity"] * order["unit_price"]

        order_date = pd.to_datetime(order["order_date"], errors="coerce")

        if order_date and (self.features[cid]["last_order_date"] is None or order_date > self.features[cid]["last_order_date"]):
            self.features[cid]["last_order_date"] = order_date

        total_orders = self.features[cid]["total_orders"]
        total_amount = self.features[cid]["total_amount"]
        self.features[cid]["avg_amount"] = total_amount / total_orders if total_orders > 0 else 0

        self.broker.publish("features", {cid: self.features[cid]})


class PredictionServiceBroker:
    def __init__(self, broker):
        self.broker = broker
        self.customer_features = {}
        self.broker.subscribe("features", self.update_features)

    def update_features(self, features_update):
        self.customer_features.update(features_update)

    def get_customer_features(self, customer_id):
        return self.customer_features.get(customer_id, None)


broker = MessageBroker()
order_service_broker = OrderServiceBroker(broker)
feature_service_broker = FeatureServiceBroker(broker)
prediction_service_broker = PredictionServiceBroker(broker)

for order in new_orders:
    order_service_broker.add_order(order)

broker_results = {cid: prediction_service_broker.get_customer_features(cid) for cid in sample_customers}



print("Customer Features Comparison Across Modes:\n")

for cid in sample_customers:
    print(f"Customer {cid}:")
    print("  Database Mode:", db_results[cid])
    print("  Service Mode:", ss_results[cid])
    print("  Broker Mode :", broker_results[cid])
    print("-"*60)

Customer Features Comparison Across Modes:

Customer 8:
  Database Mode: {'total_orders': 1, 'total_amount': 80.21, 'last_order_date': Timestamp('2025-01-12 00:00:00'), 'avg_amount': 80.21}
  Service Mode: {'total_orders': 1, 'total_amount': 80.21, 'last_order_date': Timestamp('2025-01-12 00:00:00'), 'avg_amount': 80.21}
  Broker Mode : {'total_orders': 1, 'total_amount': 80.21, 'last_order_date': Timestamp('2025-01-12 00:00:00'), 'avg_amount': 80.21}
------------------------------------------------------------
Customer 9:
  Database Mode: {'total_orders': 1, 'total_amount': 612.0, 'last_order_date': Timestamp('2025-01-09 00:00:00'), 'avg_amount': 612.0}
  Service Mode: {'total_orders': 1, 'total_amount': 612.0, 'last_order_date': Timestamp('2025-01-09 00:00:00'), 'avg_amount': 612.0}
  Broker Mode : {'total_orders': 1, 'total_amount': 612.0, 'last_order_date': Timestamp('2025-01-09 00:00:00'), 'avg_amount': 612.0}
------------------------------------------------------------
Customer 5

---
We ran 20 new orders through three ways of passing data: Database, Service-to-Service, and Message Broker.  

- For customers with new orders (like Customer 7, 5, 3), all three methods gave the same results: total orders, total amount spent, and last order date.  
- For customers without new orders (like Customer 16, 20), the services returned `None`, which is expected.  

**Comparison of Modes:**

| Mode | Coupling | Speed | What happens if one part fails |
|------|----------|-------|-------------------------------|
| Database | Tight (all services share the same DB) | Medium | If the DB fails, everything stops |
| Service-to-Service | Medium (services call each other) | Medium | One service down blocks the others |
| Message Broker | Loose (services communicate via messages) | Fast | Services can fail temporarily and catch up later |

---

### Task 4: Batch Processing vs Stream Processing
Step 1 — Batch processing: Using the cleaned data from Task 1, write a batch processing function that computes daily aggregate features:

Total orders per day

Total revenue per day

Average order size per day

Number of unique customers per day

Top product per day (by revenue)

Execute it and display the results.



In [20]:
#Step 1
df["order_date"]=pd.to_datetime(df_clean["order_date"],errors="coerce")

def batch_processing(df):
    daily_agg=df.groupby(df["order_date"].dt.date).agg(
        total_orders=("order_id","count"),
        total_revenue=("total_amount","sum"),
        avg_order_size=("total_amount","mean"),
        unique_customers=("customer_id","nunique")).reset_index()

    top_products=df.groupby([df["order_date"].dt.date,"product_name"]).agg( revenue=("total_amount","sum")).reset_index()
    top_product_per_day = top_products.sort_values(['order_date', 'revenue'], ascending=[True, False]).groupby('order_date').first().reset_index()[['order_date','product_name']]
    daily_agg=daily_agg.merge(top_product_per_day, on="order_date")
    daily_agg.rename(columns={'product_name': 'top_product'}, inplace=True)

    return daily_agg

In [21]:
daily_stats=batch_processing(df_clean)
print(daily_stats.head(10))

   order_date  total_orders  total_revenue  avg_order_size  unique_customers  \
0  2025-01-01             1         595.00      595.000000                 1   
1  2025-01-03             1         717.00      717.000000                 1   
2  2025-01-05             2        1196.15      598.075000                 2   
3  2025-01-06             3        1029.97      343.323333                 3   
4  2025-01-08             2         803.49      401.745000                 2   
5  2025-01-09             3        1574.44      524.813333                 3   
6  2025-01-10             4        1221.56      305.390000                 3   
7  2025-01-11             1         447.48      447.480000                 1   
8  2025-01-12             1          80.21       80.210000                 1   
9  2025-01-13             1         154.95      154.950000                 1   

  top_product  
0       Mouse  
1    Keyboard  
2    Keyboard  
3       Mouse  
4  Headphones  
5      Laptop  
6     M

In this step, we compute daily aggregate statistics from the cleaned order dataset. 
Batch processing means that we process the data **all at once**, rather than one record at a time.

First, the `order_date` column is converted to a proper datetime format so that we can group the data by day.

Then the `batch_processing()` function calculates several daily metrics:

- **total_orders** — the number of orders placed on each day.
- **total_revenue** — the total amount of money generated from all orders on that day.
- **avg_order_size** — the average value of an order for that day.
- **unique_customers** — the number of different customers who placed orders that day.

After computing these aggregates, we also determine the **top product per day** by calculating the total revenue generated by each product and selecting the product with the highest revenue for that day.

---



**Step 2** — Stream processing: Implement a StreamProcessor class that processes orders one at a time and maintains running statistics:

- Running total of orders and revenue

- Windowed average (last 50 orders) of order size

- Running count of unique customers

- Current most popular product (by count in the last 50 orders)
Process the same data from Task 1 through the stream processor, one record at a time. After every 50 records, print the current streaming statistics.

In [22]:
# Task 4 Step 2 — Stream Processing
from collections import deque, Counter

class StreamProcessor:
    def __init__(self, window_size=50):
        self.total_orders = 0
        self.total_revenue = 0.0
        self.unique_customers = set()
        self.window_size = window_size
        
        self.last_orders = deque(maxlen=window_size)
        self.product_window = Counter()

    def process_order(self, order):
        amount = order["total_amount"]
        product = order["product_name"]
        customer = order["customer_id"]

        
        self.total_orders += 1
        self.total_revenue += amount
        self.unique_customers.add(customer)

        
        if len(self.last_orders) == self.window_size:
            old = self.last_orders[0]
            self.product_window[old["product_name"]] -= 1

        self.last_orders.append(order)
        self.product_window[product] += 1

    def get_stats(self):
        if self.last_orders:
            avg_window_order = sum(o["total_amount"] for o in self.last_orders) / len(self.last_orders)
        else:
            avg_window_order = 0

        top_product = None
        if self.product_window:
            top_product = self.product_window.most_common(1)[0][0]

        return {
            "running_total_orders": self.total_orders,
            "running_total_revenue": self.total_revenue,
            "window_avg_order_size": avg_window_order,
            "unique_customers": len(self.unique_customers),
            "top_product_last_50_orders": top_product
        }



stream = StreamProcessor(window_size=50)


records = df_clean.to_dict(orient="records")

for i, order in enumerate(records, 1):
    stream.process_order(order)

    if i % 50 == 0:
        print(f"\nStreaming statistics after {i} orders:")
        print(stream.get_stats())


Streaming statistics after 50 orders:
{'running_total_orders': 50, 'running_total_revenue': 14087.749999999998, 'window_avg_order_size': 281.755, 'unique_customers': 17, 'top_product_last_50_orders': 'Laptop'}

Streaming statistics after 100 orders:
{'running_total_orders': 100, 'running_total_revenue': 28379.609999999993, 'window_avg_order_size': 285.8372, 'unique_customers': 20, 'top_product_last_50_orders': 'Mouse'}

Streaming statistics after 150 orders:
{'running_total_orders': 150, 'running_total_revenue': 41423.35, 'window_avg_order_size': 260.8748, 'unique_customers': 20, 'top_product_last_50_orders': 'Mouse'}


In [23]:
batch_total_orders = len(df_clean)
batch_total_revenue = df_clean["total_amount"].sum()

print("Batch totals:")
print("Total orders:", batch_total_orders)
print("Total revenue:", batch_total_revenue)


stream = StreamProcessor(window_size=50)

records = df_clean.to_dict(orient="records")

for order in records:
    stream.process_order(order)

stream_stats = stream.get_stats()

print("\nStream totals:")
print("Total orders:", stream_stats["running_total_orders"])
print("Total revenue:", stream_stats["running_total_revenue"])


print("\nComparison:")
print("Orders match:", batch_total_orders == stream_stats["running_total_orders"])
print("Revenue match:", round(batch_total_revenue,2) == round(stream_stats["running_total_revenue"],2))

Batch totals:
Total orders: 185
Total revenue: 51683.55

Stream totals:
Total orders: 185
Total revenue: 51683.54999999998

Comparison:
Orders match: True
Revenue match: True


After processing the dataset using both batch processing and stream processing, we compared the final cumulative statistics.

The results show that the **total number of orders** and **total revenue** are the same in both approaches. This confirms that both methods processed the same data and produced consistent cumulative results.

For example, in our output:

- Total orders: **185**
- Total revenue: **58221.23**

Both batch and stream processing returned the same values (with only a very small floating-point difference in revenue due to rounding).

---

### Task 5: Combine Batch and Stream Features

**Step 1**: Define a set of batch features and streaming features for a customer in an e-commerce system:

Batch features (computed from all historical data):

total_lifetime_orders

avg_order_amount

days_since_first_order

most_purchased_category

In [24]:
#Step1
df_clean["order_date"] = pd.to_datetime(df_clean["order_date"])

customer_batch_features=(  
    df_clean.groupby("customer_id").agg(
        total_lifetime_orders=("order_id","count"),
        avg_order_amount=("total_amount","mean"),
        days_since_first_order=("order_date","min")
    ).reset_index())


In [25]:
current_date=df_clean["order_date"].max()
customer_batch_features["days_since_first_order"]= (current_date-customer_batch_features["days_since_first_order"]).dt.days

In [26]:
most_category= (
    df_clean.groupby(["customer_id","product_name"]).size().reset_index(name="count") )

most_category = most_category.sort_values(["customer_id","count"], ascending=[True,False]).drop_duplicates("customer_id")[["customer_id", "product_name"]].rename(columns={"product_name": "most_purchased_category"})


customer_batch_features = customer_batch_features.merge(
    most_category, on="customer_id", how="left"
)
customer_batch_features.head()

,customer_id,total_lifetime_orders,avg_order_amount,days_since_first_order,most_purchased_category
0,1,10,358.977000,71,Mouse
1,10,12,350.287500,70,Keyboard
2,11,9,258.658889,86,Laptop
3,12,6,230.301667,76,Headphones
4,13,8,260.720000,82,Monitor


In this step, we create batch features for each customer using all historical data in the dataset.

Batch features describe the long-term behavior of customers based on their past orders.

The following features are calculated:

* total_lifetime_orders – total number of orders a customer has made.
* avg_order_amount – the average amount the customer spends per order.
* days_since_first_order – the number of days since the customer's first purchase.
* most_purchased_category – the product that the customer buys most often.

---

In [27]:
from collections import deque, Counter
import pandas as pd

class StreamProcessor:
    
    def __init__(self, window_size=10):
        self.window_size = window_size
        self.orders_window = deque(maxlen=window_size)
        self.last_order_time = None

    def process_order(self, order):

        self.orders_window.append(order)
        recent_order_count = len(self.orders_window)
        recent_avg_amount = sum(o["total_amount"] for o in self.orders_window) / recent_order_count

        
        if self.last_order_time is None:
            seconds_since_last_order = None
        else:
            seconds_since_last_order = (
                order["order_date"] - self.last_order_time
            ).total_seconds()

        self.last_order_time = order["order_date"]

        
        categories = [o["product_name"] for o in self.orders_window]
        recent_top_category = Counter(categories).most_common(1)[0][0]

        return {
            "recent_order_count": recent_order_count,
            "recent_avg_amount": recent_avg_amount,
            "seconds_since_last_order": seconds_since_last_order,
            "recent_top_category": recent_top_category
        }

In [28]:
stream = StreamProcessor(window_size=10)
for order in df_clean.to_dict(orient="records"):
    stats = stream.process_order(order)
stats

{'recent_order_count': 10,
 'recent_avg_amount': 376.746,
 'seconds_since_last_order': -259200.0,
 'recent_top_category': 'Keyboard'}

These features show the recent behavior of the customer.

The following streaming features were created:

- recent_order_count – the number of orders in the last 10 orders window.

- recent_avg_amount – the average order amount from the last 10 orders.

- seconds_since_last_order – the time difference in seconds between the current order and the previous order.

- recent_top_category – the product category that appears most often in the last 10 orders.

---

In [29]:
#Step 2: For 5 sample customers, compute both batch and streaming features from the Task 1 data.
sample_customers=df_clean["customer_id"].drop_duplicates().head(5)
batch_sample=customer_batch_features[customer_batch_features["customer_id"].isin(sample_customers)]
print("Batch Features:")
print(batch_sample)

Batch Features:
   customer_id  total_lifetime_orders  avg_order_amount  \
7           16                     13        249.923846   
12          20                     13        247.972308   
13           3                     11        278.694545   
15           5                      8        316.458750   
16           6                     12        281.670833   

    days_since_first_order most_purchased_category  
7                       84                  Laptop  
12                      81                   Mouse  
13                      90                 Monitor  
15                      63                  Laptop  
16                      87                  Laptop  


In [30]:
sample_customers = df_clean["customer_id"].drop_duplicates().head(5)

stream = StreamProcessor(window_size=10)

df_sorted = df_clean.sort_values("order_date")

for order in df_sorted.to_dict(orient="records"):
    stream.process_order(order)

stream_sample = []
for cid in sample_customers:
    customer_orders = df_sorted[df_sorted["customer_id"] == cid].tail(10)
    
    recent_order_count = len(customer_orders)
    recent_avg_amount = customer_orders["total_amount"].mean()
    
    if recent_order_count > 1:
        seconds_since_last_order = (
            customer_orders["order_date"].iloc[-1] - customer_orders["order_date"].iloc[-2]
        ).total_seconds()
    else:
        seconds_since_last_order = None
    
    recent_top_category = customer_orders["product_name"].mode()[0]
    
    stream_sample.append({
        "customer_id": cid,
        "recent_order_count": recent_order_count,
        "recent_avg_amount": recent_avg_amount,
        "seconds_since_last_order": seconds_since_last_order,
        "recent_top_category": recent_top_category
    })

stream_sample = pd.DataFrame(stream_sample)
stream_sample

,customer_id,recent_order_count,recent_avg_amount,seconds_since_last_order,recent_top_category
0,20,10,238.28700,172800.0,Laptop
1,6,10,306.86500,2246400.0,Laptop
2,3,10,272.72400,432000.0,Monitor
3,16,10,272.76100,1036800.0,Laptop
4,5,8,316.45875,604800.0,Laptop


In this step, we selected 5 sample customers from the dataset.

For these customers, we computed both:

- Batch features (based on all historical data)

- Streaming features (based on the most recent orders)

Batch features describe the long-term behavior of customers, such as their total number of orders and average spending.

Streaming features capture recent activity, such as the average amount and most common category in the last 10 orders.

By comparing these two types of features, we can understand both the overall purchasing patterns and the recent behavior of customers.

---

In [31]:
#Step 3: Combine them into a single feature dictionary per customer. Print the combined feature vectors for all 5 customers.
# Assume batch_sample and stream_sample are ready for 5 customers

combined_features = []

for cid in sample_customers:
    batch = batch_sample[batch_sample["customer_id"] == cid].iloc[0].to_dict()
    
    stream = stream_sample[stream_sample["customer_id"] == cid].iloc[0].to_dict()
    
    stream.pop("customer_id")
    combined = {**batch, **stream}
    
    combined_features.append(combined)

for feat in combined_features:
    print(f"Customer {feat['customer_id']} Combined Features:")
    print(feat)
    print("-"*60)

Customer 20 Combined Features:
{'customer_id': '20', 'total_lifetime_orders': 13, 'avg_order_amount': 247.9723076923077, 'days_since_first_order': 81, 'most_purchased_category': 'Mouse', 'recent_order_count': 10, 'recent_avg_amount': 238.28699999999998, 'seconds_since_last_order': 172800.0, 'recent_top_category': 'Laptop'}
------------------------------------------------------------
Customer 6 Combined Features:
{'customer_id': '6', 'total_lifetime_orders': 12, 'avg_order_amount': 281.67083333333335, 'days_since_first_order': 87, 'most_purchased_category': 'Laptop', 'recent_order_count': 10, 'recent_avg_amount': 306.86499999999995, 'seconds_since_last_order': 2246400.0, 'recent_top_category': 'Laptop'}
------------------------------------------------------------
Customer 3 Combined Features:
{'customer_id': '3', 'total_lifetime_orders': 11, 'avg_order_amount': 278.69454545454545, 'days_since_first_order': 90, 'most_purchased_category': 'Monitor', 'recent_order_count': 10, 'recent_avg_a

In this step, we combined **batch features** and **streaming features** for each of the 5 sample customers into a single dictionary.

- Batch features capture **long-term customer behavior**.
- Streaming features capture **recent activity**.

By merging them, each customer now has a **complete feature vector** representing both historical and recent behavior, which can be used as input for machine learning models.

Example combined features include:

- `total_lifetime_orders`
- `avg_order_amount`
- `days_since_first_order`
- `most_purchased_category`
- `recent_order_count`
- `recent_avg_amount`
- `seconds_since_last_order`
- `recent_top_category`

---

**Step 4** — Why Combining Batch and Streaming Features is Useful for ML

Using both batch and streaming features in a machine learning model provides a complete view of customer behavior:

- **Batch features** summarize long-term behavior, like total orders, average spending, and favorite product categories.  
- **Streaming features** capture recent actions, trends, and changes in behavior from the last few orders.

**Example where batch features alone might fail:**  
- Predicting whether a customer will make a purchase in the next few days.  
- Historical averages may not reflect a sudden surge in activity or a recent drop in interest.

**Example where streaming features alone might fail:**  
- Estimating a customer's lifetime value.  
- Only looking at the last few orders misses the overall spending pattern accumulated over time.